# Customer Segmentation with RFM & K-Means
## AI & ML Internship - Murick Technologies
**Author**: Muzamil Asghar
**Date**: October 08, 2025

RFM analysis and K-Means clustering on Online Retail Dataset for customer segmentation.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from datetime import datetime
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
import plotly.express as px
import plotly.graph_objects as go
%matplotlib inline
sns.set_style('whitegrid')

## 1. Load and Explore Data

In [ ]:
# Load dataset
df = pd.read_excel('Online Retail.xlsx')
print(f"Shape: {df.shape}")
display(df.head())
print(df.info())

## 2. Data Preprocessing

In [ ]:
# Clean data: Remove negatives, null CustomerID
df = df[df['Quantity'] > 0]
df = df[df['UnitPrice'] > 0]
df.dropna(subset=['CustomerID'], inplace=True)
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
print(f"Cleaned shape: {df.shape}")

## 3. RFM Analysis

In [ ]:
# Calculate RFM
snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)
rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,  # Recency
    'InvoiceNo': 'nunique',  # Frequency
    'TotalPrice': 'sum'  # Monetary
}).rename(columns={'InvoiceDate': 'Recency', 'InvoiceNo': 'Frequency', 'TotalPrice': 'Monetary'}).reset_index()

print("RFM Sample:")
display(rfm.head())
rfm.to_csv('rfm_data.csv', index=False)

In [ ]:
# RFM distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for i, col in enumerate(['Recency', 'Frequency', 'Monetary']):
    sns.histplot(rfm[col], kde=True, ax=axes[i])
    axes[i].set_title(f'{col} Distribution')
plt.tight_layout()
plt.savefig('visuals/rfm_distribution.png')
plt.show()

## 4. K-Means Clustering

In [ ]:
# Scale RFM
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm[['Recency', 'Frequency', 'Monetary']])

# Elbow Method
inertias = []
K_range = range(1, 11)
for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(rfm_scaled)
    inertias.append(kmeans.inertia_)

plt.plot(K_range, inertias, 'bo-')
plt.title('Elbow Method')
plt.xlabel('K')
plt.ylabel('Inertia')
plt.savefig('visuals/elbow.png')
plt.show()

In [ ]:
# Silhouette Score
sil_scores = []
for k in range(2, 11):
    kmeans = KMeans(n_clusters=k, random_state=42)
    labels = kmeans.fit_predict(rfm_scaled)
    sil_scores.append(silhouette_score(rfm_scaled, labels))

plt.plot(range(2, 11), sil_scores, 'ro-')
plt.title('Silhouette Score')
plt.xlabel('K')
plt.ylabel('Score')
plt.savefig('visuals/silhouette.png')
plt.show()

optimal_k = np.argmax(sil_scores) + 2
print(f"Optimal K: {optimal_k} (Silhouette: {max(sil_scores):.3f})")

In [ ]:
# Apply K-Means with optimal K=4
kmeans = KMeans(n_clusters=4, random_state=42)
rfm['Cluster'] = kmeans.fit_predict(rfm_scaled)

# Segment labels
segment_map = {0: 'Champions', 1: 'Loyal', 2: 'At-Risk', 3: 'Lost'}
rfm['Segment'] = rfm['Cluster'].map(segment_map)

print("Cluster Centers:")
centers = pd.DataFrame(scaler.inverse_transform(kmeans.cluster_centers_), columns=['Recency', 'Frequency', 'Monetary'])
display(centers.assign(Segment=segment_map.values()))

## 5. Visualizations

In [ ]:
# 3D Cluster Plot (Plotly)
fig = px.scatter_3d(rfm, x='Recency', y='Frequency', z='Monetary', color='Segment',
                    title='3D Customer Segments')
fig.write_html('interactive_3d_clusters.html')
fig.show()

In [ ]:
# Segment characteristics
segment_stats = rfm.groupby('Segment')[['Recency', 'Frequency', 'Monetary']].mean()
segment_stats.plot(kind='bar', figsize=(10, 6))
plt.title('Average RFM by Segment')
plt.savefig('visuals/segment_stats.png')
plt.show()

## 6. Key Insights
- Champions: Low Recency, High FM (15% customers, 40% revenue).
- Loyal: Medium RFM.
- At-Risk: High Recency, Medium FM.
- Lost: High Recency, Low FM.

## 7. Conclusion
Segmentation enables targeted marketing. Streamlit app for interactive exploration.